# 03: Rede Bayesiana e Manto de Markov

Etapa 3 do procedimento metodológico (Seção 3.4): aprender a estrutura da Rede Bayesiana, delimitar as variáveis
preditivas pelo Manto de Markov e testar a **H2**:

> **H2:** Espera-se que a seleção de atributos pelo Manto de Markov reduza o número de parâmetros das Tabelas de Probabilidade Condicional sem perda de desempenho preditivo.

**Fundamento do contraste.** Em uma Rede Bayesiana com as evidências representadas por estados observados, o alvo é condicionalmente independente das demais variáveis dado seu Manto. Nas redes aprendidas aqui, não há arestas saindo do alvo; o Manto corresponde aos seus pais. A rede completa e sua sub-rede ao Manto utilizam as mesmas contagens para a CPT do alvo e produzem as mesmas probabilidades neste procedimento. O estado explícito de ausência também é tratado como um estado observado.

A H2 compara a rede completa e a reduzida quanto ao número total de parâmetros livres das CPTs e ao desempenho fora da amostra. A leitura de uma CPT pequena é uma ilustração de associação preditiva, sem demonstração causal. O Naive Bayes completo e as variantes com AIC são análises de apoio, com estruturas distintas do contraste principal.

**Alcance:** avaliação retrospectiva. A validação de previsões disponíveis antes da consulta e o uso em lembretes permanecem como trabalho futuro.


**Protocolo revisado do artigo.** Na avaliação, cidades e domínios categóricos são aprendidos no treino; categorias novas viram `desconhecido`. H2 mantém ESS=1 em todas as redes para isolar a redução ao Manto; o ajuste interno de ESS pertence à H1/H3. A rede da base inteira continua apenas exploratória. Os novos resultados são salvos em `resultados/artigo_sem_sobreposicao/completo/`, preservando os históricos. H2 foi reexecutada em 15 folds; a síntese abaixo usa os resultados desta revisão.


**Correção das divisões.** Pacientes identificados e registros com os mesmos campos originais ficam em um único grupo de validação, inclusive sem ID. A assinatura inclui data e horário e exclui `no_show` e `no_show_reason`. A união dos grupos é transitiva; nenhuma linha é removida e esse identificador não entra no modelo. A regra vale também para a seleção interna dos parâmetros. Uma rodada anterior continha sobreposição de linhas e foi descartada; os resultados atuais ficam em `artigo_sem_sobreposicao/`. Isso não resolve a identificação de pessoas sem cadastro em dias diferentes.


In [ ]:
import time
import warnings

import networkx as nx
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap
from pathlib import Path

from noshow.features import CATEGORICAL, CONTINUOUS, FEATURES, Discretizer
from noshow.data import RAW_PATH, load_raw
from noshow.experiments import prepare_discrete
from noshow.preprocessing import FoldCategories
from noshow.bayes import DiscreteBN, learn_structure, markov_blanket, naive_bayes_graph
from noshow.evaluation import metrics, paired_wilcoxon, patient_groups, repeated_group_folds

warnings.filterwarnings("ignore")
import os
if os.environ.get("PYTHONHASHSEED") != "0":
    print("⚠ PYTHONHASHSEED não está fixado em 0: arestas periféricas da rede podem variar entre execuções "
          "(a estabilidade observada não é uma garantia para outras execuções). Ver README.")
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SAIDA = RAIZ / "resultados" / "artigo_sem_sobreposicao" / "completo"
SAIDA.mkdir(parents=True, exist_ok=True)

BLUE, ORANGE, GRAY, INK, GREEN = "#2a78d6", "#eb6834", "#8a8984", "#52514e", "#1baf7a"
SEQ = LinearSegmentedColormap.from_list("seq", ["#f2f1ed", ORANGE])
PCT = mtick.PercentFormatter(1, decimals=0)
plt.rcParams.update({
    "figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": GRAY, "axes.labelcolor": INK, "xtick.color": INK, "ytick.color": INK,
    "axes.grid": True, "grid.color": "#e6e5e0", "grid.linewidth": 0.6, "axes.axisbelow": True,
    "axes.titlesize": 11, "axes.titleweight": "bold", "axes.titlelocation": "left",
})

df = pd.read_parquet(RAIZ / "data" / "processed" / "features.parquet")
raw_aux = load_raw()
assert df.appointment_date.reset_index(drop=True).equals(raw_aux.appointment_date.reset_index(drop=True))
assert np.array_equal(df.no_show, raw_aux.no_show)
df["city"] = raw_aux.city.to_numpy()
raw_validation = pd.read_csv(RAW_PATH)
validation_groups = patient_groups(df.patient_id, records=raw_validation)
y = df.no_show.to_numpy()
FEATS = CATEGORICAL + CONTINUOUS
CAT_STATES = FoldCategories(CATEGORICAL).fit(df).states_  # apenas para a rede exploratória
print(f"{len(df):,} agendamentos | {len(FEATS)} variáveis candidatas | no-show {y.mean():.1%}")

## 1. Estrutura aprendida (base inteira, exploratória)

Para **visualizar e interpretar** a rede, a estrutura é aprendida uma vez com a base inteira, discretizada como no
notebook 02. A avaliação de desempenho (Seção 5) refaz tudo dentro de cada fold.

**Configuração do aprendizado:**
- **Algoritmo:** Hill Climbing com pontuação **BIC**, que penaliza a complexidade pelo tamanho da amostra.
- **No máximo 3 pais por nó:** limita o tamanho das CPTs.
- **Conhecimento especialista:** `no_show` **não pode ser pai** de nenhuma variável. A falta acontece depois que
  idade, histórico, clima e agenda já são conhecidos. Assim, o Manto de Markov do no-show são os seus pais.

> **Reprodutibilidade:** o Hill Climbing do pgmpy desempata movimentos pela ordem de iteração de conjuntos do Python,
> que depende de `PYTHONHASHSEED`. O kernel do projeto fixa essa semente em 0. Com outras sementes, de 1 a 3 arestas
> periféricas mudam, mas o Manto de Markov foi o mesmo em todas as sementes testadas.
>
> As direções das demais arestas **não devem ser lidas como causais**: dados observacionais só identificam a classe
> de equivalência de Markov. O Manto, porém, é o mesmo em toda a classe.

In [ ]:
disc_total = Discretizer(columns=CONTINUOUS).fit(df)
X = disc_total.transform(df[FEATS]).astype(str)
X["no_show"] = df.no_show.astype(str)
STATES_TOTAL = {**CAT_STATES, **disc_total.labels_, "no_show": ["0", "1"]}

t = time.time()
grafo = learn_structure(X, scoring="bic-d")
manto = markov_blanket(grafo)
print(f"Estrutura aprendida em {time.time() - t:.1f}s: {grafo.number_of_nodes()} nós, {grafo.number_of_edges()} arestas")
print("Manto de Markov do no-show (BIC):", manto)

In [ ]:
GRUPO_COR = {"demográfico": BLUE, "contextual": "#4a3aa7", "contextual (clima)": GREEN,
             "temporal": "#eda100", "comportamental": "#e87ba4"}

fig, ax = plt.subplots(figsize=(14, 10))
pos = nx.kamada_kawai_layout(grafo.to_undirected())
cores = [ORANGE if n == "no_show" else GRUPO_COR[FEATURES[n][1]] for n in grafo.nodes]
tamanhos = [1400 if n == "no_show" else 900 if n in manto else 450 for n in grafo.nodes]
arestas_manto = [(u, v) for u, v in grafo.edges if v == "no_show"]
nx.draw_networkx_edges(grafo, pos, ax=ax, edge_color="#c9c8c2", arrowsize=10, width=1,
                       edgelist=[e for e in grafo.edges if e not in arestas_manto], node_size=tamanhos)
nx.draw_networkx_edges(grafo, pos, ax=ax, edgelist=arestas_manto, edge_color=ORANGE, width=3, arrowsize=18,
                       node_size=tamanhos)
nx.draw_networkx_nodes(grafo, pos, ax=ax, node_color=cores, node_size=tamanhos, edgecolors="white", linewidths=2)
for n, (x_, y_) in pos.items():
    ax.text(x_, y_ - 0.065, n, ha="center", va="top", fontsize=8 if n not in manto + ["no_show"] else 10,
            fontweight="bold" if n in manto + ["no_show"] else "normal", color=INK)
ax.legend(handles=[plt.Line2D([], [], marker="o", ls="", ms=10, color=c, label=g) for g, c in GRUPO_COR.items()]
          + [plt.Line2D([], [], marker="o", ls="", ms=12, color=ORANGE, label="no_show (alvo)")],
          frameon=False, fontsize=8, loc="lower left")
ax.set_title("Rede Bayesiana aprendida (BIC) — arestas laranja: Manto de Markov do no-show")
ax.axis("off"); plt.tight_layout(); plt.show()

## 2. O Manto de Markov e por que as outras variáveis saem

Das 25 variáveis candidatas, apenas `taxa_faltas_anteriores` e `periodo` ficam no Manto (BIC). As outras não são
irrelevantes: **dado o Manto, o que elas acrescentam não compensa o custo em parâmetros**. Um exemplo concreto é
`cadastro_ausente`, que sozinha separa 15,1% de 9,1% de no-show (notebook 02). Um paciente sem cadastro também não
tem ID, logo sua `taxa_faltas_anteriores` já está no estado "ausente": **a maior parte** da informação já entrou na
rede por esse caminho. O que sobra (abaixo) é uma diferença menor, dentro do grupo "ausente".

In [ ]:
print(pd.crosstab(df.cadastro_ausente.map({"0": "com cadastro", "1": "sem cadastro"}),
                  X.taxa_faltas_anteriores == "ausente", rownames=[""], colnames=["taxa_faltas_anteriores = ausente"]))

taxa_ausente = X.taxa_faltas_anteriores == "ausente"
cond = df[taxa_ausente].groupby("cadastro_ausente").no_show.agg(["mean", "size"])
print("\nDentro de 'taxa_faltas_anteriores = ausente', a taxa de no-show por cadastro_ausente:")
print(cond.rename(index={"0": "com cadastro", "1": "sem cadastro"}).round(3).to_string())

Essa diferença residual vale a pena para o modelo? O BIC responde. A célula abaixo calcula a variação da pontuação
ao acrescentar cada variável como **terceiro pai** do no-show. Valores negativos significam que o ganho de ajuste não
paga os parâmetros extras. Isso confirma que a exclusão é uma decisão do critério, e não uma limitação da busca do
Hill Climbing.

In [ ]:
from pgmpy.estimators import BIC

bic = BIC(X)
base_score = bic.local_score("no_show", manto)
delta = pd.Series({v: bic.local_score("no_show", manto + [v]) - base_score
                   for v in FEATS if v not in manto}).sort_values(ascending=False)
print(f"Variação do BIC ao acrescentar um 3º pai (máximo: {delta.max():+.1f}, todas negativas: {(delta < 0).all()})")
delta.head(8).round(1).to_frame("Δ BIC")

**Sensibilidade ao critério de pontuação.** BIC penaliza mais a complexidade; AIC, menos; BDeu e K2 são critérios
bayesianos. Se o Manto mudasse muito entre eles, a seleção seria frágil.

In [ ]:
sens = []
for nome, score in [("BIC", "bic-d"), ("AIC", "aic-d"), ("BDeu", "bdeu"), ("K2", "k2")]:
    g = learn_structure(X, scoring=score)
    mb = markov_blanket(g)
    bn = DiscreteBN(g).fit(X, STATES_TOTAL)
    sens.append({"critério": nome, "Manto de Markov": ", ".join(mb), "nº variáveis": len(mb),
                 "arestas na rede": g.number_of_edges(), "parâmetros da CPT do no-show": bn.n_params(["no_show"])})
pd.DataFrame(sens).set_index("critério")

O núcleo `taxa_faltas_anteriores` + `periodo` aparece em **todos** os critérios. O AIC acrescenta `mes`, e o K2
acrescenta `faltou_ultima`, que é uma versão resumida do mesmo histórico. Ou seja, o comportamento passado do paciente
e o momento do serviço (a deriva temporal) bastam para explicar o que as 25 variáveis explicam juntas.

## 3. Capacidade explicativa: a CPT do no-show

Com o Manto (BIC), toda a "lógica" do modelo cabe em uma tabela das faixas de histórico × três períodos observados (os estados reservados são omitidos da figura): a probabilidade de falta para cada
combinação de histórico e período. Nenhum Naive Bayes com 25 variáveis permite essa leitura direta.

In [ ]:
bn_total = DiscreteBN(grafo).fit(X, STATES_TOTAL)
cpt = bn_total.cpt_frame("no_show")["1"].unstack("periodo")
ordem = [s for s in disc_total.labels_["taxa_faltas_anteriores"] if s in cpt.index]
cpt = cpt.loc[ordem, ["pre_pandemia", "pandemia", "pos_2022"]]
n = X.groupby(["taxa_faltas_anteriores", "periodo"]).size().unstack().reindex_like(cpt)

fig, ax = plt.subplots(figsize=(8, 4.2))
sns.heatmap(cpt, annot=cpt.map(lambda v: f"{v:.1%}") + "\n" + n.map(lambda v: f"n={v:,.0f}"), fmt="", cmap=SEQ,
            cbar=False, linewidths=2, linecolor="white", ax=ax, annot_kws={"size": 9})
ax.set_title("P(no-show | taxa de faltas anteriores, período)")
ax.set_xlabel(""); ax.set_ylabel("taxa de faltas anteriores")
plt.tight_layout(); plt.show()

**Leitura:** dentro de cada período, o risco sobe com a taxa de faltas anteriores. Entre períodos, o patamar inteiro
sobe, que é a deriva temporal. Pacientes sem histórico identificável ("ausente") ficam em um patamar alto, porque
reúnem os sem cadastro e os recém-chegados. Para um sistema de mensageria, essa tabela é a regra de decisão
**auditável**: "paciente com mais de 14% de faltas no período atual → risco de X%".

**Cuidado com as células pequenas.** Combinações com poucos registros (2022 tem menos de 100 em quase todas as
faixas; a faixa de 0–3% na pandemia tem 62) produzem estimativas instáveis e quebram o padrão crescente, como os 0,2%
de 2022 com n = 16. A priori BDeu evita zeros, mas, com tamanho equivalente de 1, suaviza pouco. Em produção, vale
um tamanho equivalente maior ou agrupar os períodos com pouca amostra.

## 4. Parcimônia

A rede completa precisa de parâmetros para **todas** as 26 variáveis (cada uma condicionada aos seus pais). A rede
reduzida ao Manto só precisa da CPT do no-show e das distribuições do próprio Manto.

In [ ]:
rede_manto = DiscreteBN(grafo.subgraph(manto + ["no_show"]).copy()).fit(X, STATES_TOTAL)
nb_completo = DiscreteBN(naive_bayes_graph(FEATS)).fit(X, STATES_TOTAL)
parcimonia = pd.DataFrame({
    "variáveis na rede": [len(FEATS), len(manto), len(FEATS)],
    "variáveis usadas na predição": [len(manto), len(manto), len(FEATS)],
    "parâmetros livres (rede inteira)": [bn_total.n_params(), rede_manto.n_params(), nb_completo.n_params()],
}, index=["RB completa (BIC)", "RB reduzida ao Manto (BIC)", "Naive Bayes completo"])
parcimonia["redução vs RB completa"] = parcimonia.iloc[0, 2] / parcimonia.iloc[:, 2]
parcimonia.style.format({"parâmetros livres (rede inteira)": "{:,}", "redução vs RB completa": "{:.0f}×"})

## 5. Experimento controlado (H2)

**Desenho:**
- **Validação cruzada repetida** (5 folds × 3 repetições = 15 pares), **agrupada por paciente e por registros originais indistinguíveis**: cada grupo fica apenas no treino ou no teste; pessoas sem ID em dias diferentes não têm separação garantida. O Wilcoxon é apresentado como análise exploratória (as repetições medem estabilidade; os folds compartilham observações e não são amostras independentes).
- **Em cada fold:** a discretização é ajustada no treino, a estrutura é aprendida no treino (BIC e AIC), o Manto é
  extraído e as CPTs são estimadas no treino com priori BDeu. A avaliação é feita no teste.

**Modelos comparados:**

| Modelo | O que é |
|---|---|
| `NB_completo` | **controle:** Naive Bayes com as 25 variáveis, sem filtro |
| `RB_completa_*` | rede aprendida com as 25 variáveis |
| `RB_manto_*` | **experimental:** mesma rede reduzida ao Manto de Markov |
| `NB_manto_*` | Naive Bayes só com as variáveis do Manto (separa o efeito da seleção do efeito da estrutura) |

**Métricas:** ROC-AUC, PR-AUC (critério principal, Seção 2.3.4) e **Brier**, o erro quadrático das probabilidades:
quanto menor, menor o erro probabilístico; Brier combina aspectos de calibração e discriminação. Essa última importa porque a mensageria vai agir sobre a probabilidade, e não só
sobre a ordenação dos pacientes.

In [ ]:
resultados, mantos = [], []
t0 = time.time()
for r, f, tr, te in repeated_group_folds(y, validation_groups, n_splits=5, n_repeats=3):
    Xtr, Xte, estados = prepare_discrete(df.iloc[tr], df.iloc[te], CATEGORICAL, CONTINUOUS)
    Xtr["no_show"] = y[tr].astype(str)

    for crit, score in [("bic", "bic-d"), ("aic", "aic-d")]:
        g = learn_structure(Xtr, scoring=score)                     # estrutura só no treino
        mb = markov_blanket(g)
        mantos.append({"rep": r, "fold": f, "critério": crit.upper(), "manto": ", ".join(mb)})
        modelos = {f"RB_completa_{crit}": g, f"RB_manto_{crit}": g.subgraph(mb + ["no_show"]).copy(),
                   f"NB_manto_{crit}": naive_bayes_graph(mb)}
        if crit == "bic":
            modelos["NB_completo"] = naive_bayes_graph(FEATS)
        for nome, gr in modelos.items():
            bn = DiscreteBN(gr).fit(Xtr[list(gr.nodes)], estados)
            t = time.perf_counter()
            p = bn.predict_proba(Xte)[:, 1]
            ms_10k = (time.perf_counter() - t) / len(te) * 1e7
            resultados.append({"rep": r, "fold": f, "modelo": nome, "n_vars": len(markov_blanket(gr)),
                               "n_params": bn.n_params(), "ms_por_10k": ms_10k, **metrics(y[te], p)})

R = pd.DataFrame(resultados)
R.to_csv(SAIDA / "h2_validacao_cruzada.csv", index=False)
pd.DataFrame(mantos).to_csv(SAIDA / "h2_mantos.csv", index=False)
print(f"15 folds × 7 modelos em {time.time() - t0:.0f}s → {SAIDA / 'h2_validacao_cruzada.csv'}")

In [ ]:
mantos_df = pd.DataFrame(mantos)
frequencia = (mantos_df.assign(manto=mantos_df.manto.str.split(", ")).explode("manto").groupby(["critério", "manto"]).size()
              .rename("folds").reset_index())
frequencia["frequencia"] = frequencia["folds"] / mantos_df.groupby("critério").size().reindex(frequencia["critério"]).to_numpy()
frequencia.to_csv(SAIDA / "h2_frequencia_variaveis.csv", index=False)
print("Frequência de inclusão de cada variável no Manto, em 15 folds por critério:")
frequencia.sort_values(["critério", "frequencia"], ascending=[True, False])

In [ ]:
ORDEM = ["NB_completo", "RB_completa_bic", "RB_manto_bic", "NB_manto_bic", "RB_completa_aic", "RB_manto_aic", "NB_manto_aic"]
resumo = R.groupby("modelo").agg(
    roc_auc=("roc_auc", "mean"), roc_dp=("roc_auc", "std"), pr_auc=("pr_auc", "mean"), pr_dp=("pr_auc", "std"),
    brier=("brier", "mean"), n_vars=("n_vars", "mean"), n_params=("n_params", "mean"), ms_por_10k=("ms_por_10k", "median"),
).loc[ORDEM]
resumo.style.format({"roc_auc": "{:.4f}", "roc_dp": "±{:.4f}", "pr_auc": "{:.4f}", "pr_dp": "±{:.4f}",
                     "brier": "{:.4f}", "n_vars": "{:.0f}", "n_params": "{:,.0f}", "ms_por_10k": "{:.1f}"}) \
    .highlight_max(subset=["roc_auc", "pr_auc"], color="#d6e6f8").highlight_min(subset=["brier"], color="#d6e6f8")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4.2), sharey=True)
ypos = {m: i for i, m in enumerate(ORDEM[::-1])}
for ax, (met, titulo) in zip(axes, [("pr_auc", "PR-AUC (↑)"), ("roc_auc", "ROC-AUC (↑)"), ("brier", "Brier (↓)")]):
    for m in ORDEM:
        v = R.loc[R.modelo == m, met]
        cor = ORANGE if m.startswith("RB_manto") else GRAY if m == "NB_completo" else BLUE
        ax.scatter(v, np.full(len(v), ypos[m]) + np.random.default_rng(0).uniform(-0.15, 0.15, len(v)),
                   s=14, color=cor, alpha=0.6, linewidths=0)
        ax.plot(v.mean(), ypos[m], "|", color=INK, ms=16, mew=2)
    ax.set_title(titulo); ax.grid(axis="y", visible=False)
axes[0].set_yticks(list(ypos.values()), list(ypos.keys()))
fig.suptitle("Desempenho nos 15 folds (pontos = folds, traço = média)", x=0.01, ha="left", fontweight="bold")
plt.tight_layout(); plt.show()

**Verificação da teoria:** a rede completa e a rede reduzida ao Manto devem dar **predições idênticas** em todos os
folds.

In [ ]:
for crit in ["bic", "aic"]:
    for met in ["roc_auc", "pr_auc", "brier"]:
        a = R[R.modelo == f"RB_completa_{crit}"][met].to_numpy()
        b = R[R.modelo == f"RB_manto_{crit}"][met].to_numpy()
        assert np.allclose(a, b), (crit, met)
print("✓ RB completa e RB reduzida ao Manto têm métricas idênticas em todos os folds (BIC e AIC)")

**Testes pareados de Wilcoxon** (15 pares por comparação, alternativa bilateral). `diferença_mediana` = modelo da
esquerda − modelo da direita. `pares_a_maior` conta em quantos folds o da esquerda teve valor maior; no Brier, menor é
melhor.

In [ ]:
comparacoes = [("RB_manto_bic", "NB_completo"), ("RB_manto_aic", "NB_completo"),
               ("RB_manto_aic", "RB_manto_bic"), ("RB_manto_bic", "NB_manto_bic")]
testes = []
for a, b in comparacoes:
    for met in ["pr_auc", "roc_auc", "brier"]:
        piv = R.pivot_table(index=["rep", "fold"], columns="modelo", values=met)
        testes.append({"comparação": f"{a} × {b}", "métrica": met, **paired_wilcoxon(piv[a], piv[b])})
testes = pd.DataFrame(testes).set_index(["comparação", "métrica"])
testes.style.format({"diferença_mediana": "{:+.4f}", "p": "{:.4f}"}) \
    .map(lambda v: "font-weight: bold" if v < 0.05 else "", subset=["p"])

## 6. Sensibilidade: o limite de pais e o papel do Noisy-OR

O limite de 3 pais por nó poderia estar "cortando" o Manto. A tabela abaixo relaxa esse limite até 5 pais. Ela
também mostra o custo de cada Manto: a CPT completa cresce **multiplicativamente** com os estados dos pais, enquanto
o **Noisy-OR**, ao supor causas independentes, cresce **linearmente**.

In [ ]:
lim = []
for max_pais in [2, 3, 4, 5]:
    for nome, score in [("BIC", "bic-d"), ("AIC", "aic-d")]:
        g = learn_structure(X, scoring=score, max_indegree=max_pais)
        mb = markov_blanket(g)
        estados = [len(STATES_TOTAL[v]) for v in mb]
        lim.append({"máx. de pais": max_pais, "critério": nome, "Manto": ", ".join(mb),
                    "parâmetros da CPT (tabela completa)": int(np.prod(estados)) if mb else 1,
                    "parâmetros com Noisy-OR (≈ Σ estados)": int(sum(estados)) + 1})
pd.DataFrame(lim).set_index(["máx. de pais", "critério"]).style.format(
    {"parâmetros da CPT (tabela completa)": "{:,}", "parâmetros com Noisy-OR (≈ Σ estados)": "{:,}"})

O Manto **não cresce** com o limite relaxado, nem com o BIC (2 variáveis) nem com o AIC (3): a amostra não sustenta
mais pais, e o limite de 3 não foi o que definiu o resultado. O Noisy-OR, portanto, **não é necessário para a
eficiência** deste modelo, já que a CPT do alvo permanece pequena, conforme a contagem da tabela. Ele
passaria a ser útil se o critério fosse mais permissivo ou se novas variáveis (como a data de marcação) aumentassem
o Manto.

## 7. Síntese: H2

In [ ]:
res = resumo
nb, mbic, maic = res.loc["NB_completo"], res.loc["RB_manto_bic"], res.loc["RB_manto_aic"]
rc = res.loc["RB_completa_bic"]
print(f"Parcimônia: RB reduzida ao Manto (BIC) usa {mbic.n_vars:.0f} variáveis e {mbic.n_params:,.0f} parâmetros, "
      f"contra {rc.n_params:,.0f} da RB completa ({rc.n_params / mbic.n_params:.0f}× menos), com predições idênticas.")
print(f"PR-AUC: NB completo {nb.pr_auc:.4f} | RB Manto BIC {mbic.pr_auc:.4f} | RB Manto AIC {maic.pr_auc:.4f}")
print(f"ROC-AUC: NB completo {nb.roc_auc:.4f} | RB Manto BIC {mbic.roc_auc:.4f} | RB Manto AIC {maic.roc_auc:.4f}")
print(f"Brier: NB completo {nb.brier:.4f} | RB Manto BIC {mbic.brier:.4f} | RB Manto AIC {maic.brier:.4f}")

**Síntese de H2 após corrigir os grupos.**

A avaliação usa os grupos conjuntos de paciente e registros originais indistinguíveis, com zero sobreposição auditada. As métricas da rede completa e da reduzida são idênticas em todos os 15 folds, com BIC e AIC.

| Modelo | PR-AUC | ROC-AUC | Brier |
|---|---|---|---|
| NB completo | 0,1398 | 0,6287 | 0,1664 |
| RB Manto BIC | 0,1432 | 0,6190 | 0,0865 |
| RB Manto AIC | 0,1576 | 0,6319 | 0,0863 |

Com BIC, a média de parâmetros cai de 12642,0 para 39,0 (aproximadamente 324 vezes menos). A frequência por variável nos 15 folds está na tabela de estabilidade. Esse resultado sustenta parcimônia e legibilidade estrutural; não estabelece causalidade nem foi submetido a avaliação de compreensão por profissionais do serviço.

Os pacientes sem ID em consultas diferentes continuam sem identificação garantida. Os testes sobre folds repetidos são exploratórios. O Noisy-OR é analisado na Seção 6 apenas como comparação teórica neste notebook; a implementação utilizou CPTs completas com BDeu e a discussão sobre Noisy-OR foi retirada do artigo conforme orientação.
